# Chapter 2: LLM API Basics

In [1]:
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from litellm import completion
from pydantic import BaseModel, Field

load_dotenv(find_dotenv())

True

## Listing 2.2: Basic OpenAI API Call

In [2]:
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)   

In [3]:
print(response.choices[0].message.content)

The capital of France is Paris.


### Experiment: API can write several independent answers to the same prompt in one request

In [4]:
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Suggest a name for a hotel room-service bot."},
    ],
    n=3
)

for i, choice in enumerate(response.choices):
    print(i, choice.finish_reason, choice.message.content)
    
print(response.usage)

0 stop Here are some name ideas, grouped by tone. Short tagline included so you can picture the personality.

Luxury / Elegant
- Orchid — refined, calming service
- Velvet — smooth, discreet help
- Lumo — soft, modern luxury
- Tranquil — restful, concierge-like

Friendly / Warm
- SuiteMate — your in-room companion
- PillowPal — cozy, approachable assistance
- Hostly — friendly, helpful host
- ParcelPal — practical and personable

Butler / Concierge-style
- RoomButler — classic, reliable service
- ConciergeOne — professional, single-touch help
- Minerva — wise, attentive concierge
- KeyMate — personal, on-demand assistant

Tech-forward / Modern
- EchoServe — responsive, voice-ready bot
- ServeSense — smart, anticipatory service
- Quiki — snappy, efficient requests
- Halo — simple, clean interface

Playful / Charming
- NiteOwl — night-service specialist
- Turndown — charmingly on-brand for hotels
- NimbleNook — quick, clever helper
- BellaServe — friendly with a touch of warmth

Top 3 qu

## Listing 2.5: Demonstrating API Statelessness

In [5]:
# First call
response1 = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is Sharon."}]
)

#Second call
response2 = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "What is my name?"}]
)

In [6]:
print(response2.choices[0].message.content)

I don't know — I don't have access to your personal data unless you tell me. What would you like me to call you? (You can reply, for example, "Call me Alex" or "My name is Maria".)


## Listing 2.6: Managing conversation history

In [7]:
messages = []

#First exchange
messages.append({"role": "user", "content": "My name is Sharon!"})
response1 = completion(model="gpt-5-mini", messages=messages) 

In [8]:
assistant_message1 = response1.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_message1})

In [9]:
print(assistant_message1)

Hi Sharon — nice to meet you! How can I help you today?


In [10]:
#Second exchange - includes previous conversation history
messages.append({"role": "user", "content": "What is my name?"})
response2 = completion(model="gpt-5-mini", messages=messages)
assistant_message2 = response2.choices[0].message.content

In [11]:
print(assistant_message2)

Your name is Sharon.


In [12]:
print(response2.usage.prompt_tokens)

41


In [13]:
print(response1.usage.prompt_tokens)

11


## Listing 2.7: Structured output with Pydantic

In [14]:
class ExtractedInfo(BaseModel):
    name: str
    email: str
    phone: str | None = None

In [15]:
response = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is John Smith, my email is John@example.com, and my phone is 555-1234"}],
    response_format=ExtractedInfo
)

In [16]:
result = response.choices[0].message.content

In [17]:
print(type(result))

<class 'str'>


In [18]:
info = ExtractedInfo.model_validate_json(result)

In [19]:
info.email

'John@example.com'

In [20]:
info.phone

'555-1234'

### Experiment: What happens when the one field in the schema is missing?

In [21]:
response = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is John Smith, my email is John@example.com"}],
    response_format=ExtractedInfo
)

In [22]:
result = response.choices[0].message.content

In [23]:
info = ExtractedInfo.model_validate_json(result)

In [24]:
info.phone

In [25]:
print(result)

{"name":"John Smith","email":"John@example.com","phone":null}


In [26]:
response = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is John Smith"}],
    response_format=ExtractedInfo
)

In [27]:
result = response.choices[0].message.content

In [28]:
info = ExtractedInfo.model_validate_json(result)

In [29]:
info.email

''

In [30]:
print(type(info))

<class '__main__.ExtractedInfo'>


In [31]:
info.phone

### Learning: Pydantic only errors when the key is missing or the value is the wrong type.

In [33]:
from pydantic import ValidationError
try:
    ExtractedInfo.model_validate_json('{"name": "A"}')
except ValidationError as e:
    print(e)

1 validation error for ExtractedInfo
email
  Field required [type=missing, input_value={'name': 'A'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [34]:
try:
    ExtractedInfo.model_validate_json('{"name": "A", "email": 5}')
except ValidationError as e:
    print(e)

1 validation error for ExtractedInfo
email
  Input should be a valid string [type=string_type, input_value=5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type


### Experiment: Use Field(...) to attach extra rules to a field

In [35]:
class ExtractedInfo(BaseModel):
    name: str
    email: str = Field(min_length=1)
    phone: str | None = None

In [36]:
try:
    ExtractedInfo.model_validate_json('{"name": "A", "email": ""}')
except ValidationError as e:
    print(e)

1 validation error for ExtractedInfo
email
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short


### Experiment: Why making a field optional is usually the better design when the data might genuinely be missing?

In [37]:
class ExtractedInfo(BaseModel):
    name: str
    email: str | None = None
    phone: str | None = None

In [38]:
for i in range(5):
    response = completion(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": "My name is John Smith"}],
        response_format=ExtractedInfo
    )
    info = ExtractedInfo.model_validate_json(response.choices[0].message.content)
    print(i, info.email)

0 None
1 None
2 None
3 None
4 None


In [39]:
for i in range(5):
    response = completion(
        model="gpt-5-mini",
        messages=[{"role": "system", "content": "Only extract values stated in the text. If a field isn't mentioned, use null. Never guess."},{"role": "user", "content": "My name is John Smith"}],
        response_format=ExtractedInfo
    )
    info = ExtractedInfo.model_validate_json(response.choices[0].message.content)
    print(i, info.email)

0 None
1 None
2 None
3 None
4 None


1. The schema change mattered: required str gave '', and optional gave None every time. Just giving the model a legal “missing” answer changed its behavior.
2. The instruction didn’t matter here, because the case was too easy. Prompts make a difference on tempting inputs, like “email me at my usual Gmail” or “it’s my name at the company domain”. There, a guess looks reasonable.
3. 5/5 isn’t proof: a model that fails 1 time in 10 would still often pass 5 in a row. Real evals use dozens to hundreds of cases, which is why GAIA uses 20 problems even in the quick version. You’ll see this again in Ch10.